# Nitrogen + n-C9: symbolic discovery and VLE comparison

Run from top to bottom. Provide your own input files first (see `data/README.md`). No experimental data or predictions are bundled. The plot uses one temperature panel per isotherm. Search and baseline recalculation are opt-in. The proposed model is trained jointly on all six systems in `00_shared_model.ipynb`, not independently here.

**Data required.** This code-only release contains no measurements, predictions or example figures. Prepare inputs using `data/README.md` before running the cells. For a software-only check, generate explicitly fictional inputs with `python -m script.synthetic_demo` as described in the main README.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "configs/search_settings.json").exists():
    ROOT = ROOT.parent
if not (ROOT / "configs/search_settings.json").exists():
    raise RuntimeError("Open this notebook from the repository or notebooks directory.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from script import workflow
from script.data import load_preos_predictions
from script.plotting import plot_system
from IPython.display import display, Image
print("Repository:", ROOT)


In [ ]:
SYSTEM = "C9N2"
RUN_DISCOVERY = False
RECOMPUTE_BASELINES = False
SEED = 1
PREDICTIONS = ROOT / "outputs/proposed_predictions.csv"


## Inspect the calibrated baseline
Pressure is in MPa. Both composition columns are nitrogen mole fractions.

In [ ]:
baseline, pressure_column = load_preos_predictions(SYSTEM)
display(baseline.head())
print("Available calibrated states:", len(baseline))


## Optional system-specific symbolic searches
Set `RUN_DISCOVERY = True` above to run the manuscript search configuration. This may take substantially longer than plotting. Candidate expressions, normalization constants and settings are saved in a new output directory. These searches do not automatically replace the manually selected shared basis templates.

In [ ]:
if RUN_DISCOVERY:
    discovery_dir = workflow.discover(SYSTEM, seed=SEED)
    print(discovery_dir)


## Optional baseline recalculation
User-provided baseline files are used by default. Recalculation preserves failed states as NaN and writes a solver report. Set `BASELINE_DIRS` to the generated directories to plot these instead of the baseline files you prepared.

In [ ]:
BASELINE_DIRS = {}
if RECOMPUTE_BASELINES:
    for kind in ["zero", "calibrated", "generalized"]:
        BASELINE_DIRS[kind] = workflow.recompute_baselines([SYSTEM], kind=kind)


## Temperature-panel VLE comparison
Experiment, PR-EOS with zero BIP, calibrated PR-EOS, the generalized Yang BIP baseline and Two-level SR are shown. For a newly trained model, change `PREDICTIONS` to its `predictions.csv`. A separate CSV logs plotting exclusions, without modifying the source predictions.

In [ ]:
figure = plot_system(SYSTEM, predictions=PREDICTIONS,
                     output_dir=ROOT / "outputs/figures", baseline_dirs=BASELINE_DIRS)
display(Image(filename=str(figure.with_suffix(".png"))))
print("Vector PDF:", figure)
